# Детекция заправок: улучшение событийной метрики

**Цель задания** — увеличить **событийный F1** на тестовых машинах.
Бейзлайн из исходного ноутбука — **0.786**, планка «хорошо» — выше **0.90**.

Этот ноутбук воспроизводит бейзлайн и добавляет к нему систему улучшений,
подобранных **только на train-машинах** (валидация — часть train, тест не трогаем).

## Что улучшаем и почему

Метрика событийная: оператору нужен факт «заправился», а не разметка каждого
5-минутного интервала. Поэтому основная ставка — не только архитектура, но и
**постобработка** предсказаний (порог, минимальная длина события, склейка разрывов).

План:

1. **Признаки.** Добавляем к `fuel, d1..d4` разности `d6, d12`, признаки
   `ign, speed, n_msg`, время суток и локальную статистику уровня.
2. **Нормализация.** Стандартизация по статистикам **только обучающих машин**.
3. **Окно и модель.** Окно 9 интервалов, двунаправленная LSTM + attention.
4. **Постобработка.** Порог по вероятности класса 1 + минимальная длина события +
   склейка коротких разрывов. Гиперпараметры выбираются на валидации.

Архитектура и функции метрик взяты из исходного ноутбука, чтобы результат
воспроизводился 1:1.

## 1. Данные и общие утилиты

In [ ]:
import os
import time
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             classification_report)

torch.manual_seed(42)
np.random.seed(42)

DATA_URL = "https://raw.githubusercontent.com/hehehelp123/fuel-refuel-lab/main/data/fuel_5min.csv"
LOCAL = "fuel_5min.csv"
if not os.path.exists(LOCAL):
    print("качаю датасет...")
    urllib.request.urlretrieve(DATA_URL, LOCAL)

df = pd.read_csv(LOCAL, parse_dates=["ts"])
print("строк:", len(df), "| машин:", df.vehicle.nunique())
print("метки:", df.label.value_counts().sort_index().to_dict())
print("split:", df.split.value_counts().to_dict())

### Утилиты метрик (без изменений из исходного ноутбука)

Событийный F1 считаем так же: блоки класса 1 сопоставляются с допуском ±5 минут,
попадание в класс 2 не штрафуется, событие засчитывается один раз.

In [ ]:
def count_events(labels, positive=1):
    arr = np.asarray(labels) == positive
    padded = np.pad(arr, (1, 1), constant_values=False)
    return len(np.where(np.diff(padded))[0]) // 2


def blocks(mask, times):
    out, start = [], None
    for i, v in enumerate(mask):
        if v and start is None:
            start = i
        elif not v and start is not None:
            out.append((times[start], times[i - 1]))
            start = None
    if start is not None:
        out.append((times[start], times[len(mask) - 1]))
    return out


TOLERANCE = np.timedelta64(5, "m")


def event_metrics(y_true, y_pred, meta, verbose=False):
    n_true = n_true_hit = 0
    n_pred = n_pred_hit = 0
    for v, sub in meta.groupby("vehicle"):
        pos = sub.index.to_numpy()
        times = sub["ts"].values
        t_true = blocks(y_true[pos] == 1, times)
        t_pred = blocks(y_pred[pos] == 1, times)
        t_amb = blocks(y_true[pos] == 2, times)
        hit_true = set()
        for ps, pe in t_pred:
            hit = [k for k, (ts_, te_) in enumerate(t_true)
                   if ps <= te_ + TOLERANCE and pe >= ts_ - TOLERANCE]
            if hit:
                hit_true.update(hit)
                n_pred += 1
                n_pred_hit += 1
            elif any(ps <= te_ + TOLERANCE and pe >= ts_ - TOLERANCE
                     for ts_, te_ in t_amb):
                pass
            else:
                n_pred += 1
        n_true += len(t_true)
        n_true_hit += len(hit_true)
    tp, fn = n_true_hit, n_true - n_true_hit
    fp = n_pred - n_pred_hit
    precision = n_pred_hit / n_pred if n_pred else 0.0
    recall = n_true_hit / n_true if n_true else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    if verbose:
        print(f"события: TP={tp} FP={fp} FN={fn}")
        print(f"precision={precision:.3f} recall={recall:.3f} F1={f1:.3f}")
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision,
            "recall": recall, "f1": f1}


def report(y_true, y_pred, meta, title="модель"):
    p = precision_score(y_true, y_pred, labels=[1], average="macro", zero_division=0)
    r = recall_score(y_true, y_pred, labels=[1], average="macro", zero_division=0)
    f = f1_score(y_true, y_pred, labels=[1], average="macro", zero_division=0)
    macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    ev = event_metrics(y_true, y_pred, meta)
    print(f"=== {title} ===")
    print(f"точечные (класс 1): precision={p:.3f}  recall={r:.3f}  F1={f:.3f}")
    print(f"macro F1 по трём классам: {macro:.3f}")
    print(f"событийные: TP={ev['tp']} FP={ev['fp']} FN={ev['fn']}  "
          f"precision={ev['precision']:.3f} recall={ev['recall']:.3f} F1={ev['f1']:.3f}")
    return {"point_f1": f, "event_f1": ev["f1"]}

## 2. Признаки

Базовый набор — `fuel, d1..d4`. Добавляем:

* разности на длинном горизонте `d6, d12` — для медленной заправки, размазанной на час;
* `ign, ign_on` — заправка с включённым двигателем почти всегда идёт при зажигании;
* `speed, speed_low` — на заправке машина стоит;
* `n_msg_c` — плотность сообщений (косвенно — качество сигнала);
* циклическое время суток `hour_sin/cos` и `is_day`;
* локальную статистику уровня `fuel_roll3/7`, отклонение от сглаженного `fuel_resid`,
  разброс `fuel_roll_std5`, локальные экстремумы прироста `d1_max3/d1_min3`.

Признаки считаются **внутри каждой машины** — окна разных ТС не склеиваются.

In [ ]:
ALL_FEATURES = ["fuel", "d1", "d2", "d3", "d4", "d6", "d12",
                "ign", "ign_on", "speed", "speed_low", "n_msg_c",
                "hour_sin", "hour_cos", "is_day",
                "fuel_roll3", "fuel_roll7", "fuel_resid", "fuel_roll_std5",
                "d1_max3", "d1_min3"]
BASE_FEATURES = ["fuel", "d1", "d2", "d3", "d4"]


def add_features(g):
    g = g.sort_values("ts").copy()
    f = g["fuel"]
    for k in (1, 2, 3, 4, 6, 12):
        g[f"d{k}"] = f.diff(k).fillna(0.0)
    g["ign_on"] = (g["ign"] > 0.5).astype(float)
    g["speed_low"] = (g["speed"] < 3).astype(float)
    hour = g["ts"].dt.hour + g["ts"].dt.minute / 60.0
    g["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    g["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    g["is_day"] = ((hour >= 6) & (hour <= 22)).astype(float)
    g["n_msg_c"] = g["n_msg"].clip(upper=60) / 60.0
    g["fuel_roll3"] = f.rolling(3, center=True, min_periods=1).mean()
    g["fuel_roll7"] = f.rolling(7, center=True, min_periods=1).mean()
    g["fuel_resid"] = f - g["fuel_roll7"]
    g["fuel_roll_std5"] = f.rolling(5, center=True, min_periods=1).std().fillna(0.0)
    g["d1_max3"] = g["d1"].rolling(3, center=True, min_periods=1).max()
    g["d1_min3"] = g["d1"].rolling(3, center=True, min_periods=1).min()
    return g


df = pd.concat([add_features(g) for _, g in df.groupby("vehicle")], ignore_index=True)

train_veh = sorted(df[df.split == "train"].vehicle.unique())
ev_per_veh = {v: count_events(df[df.vehicle == v].label) for v in train_veh}
VAL_VEH = set(sorted(train_veh, key=lambda v: -ev_per_veh[v])[1::5][:9])
print("машин train:", len(train_veh), "| в валидации:", len(VAL_VEH))

## 3. Нормализация и окна

Стандартизуем признаки статистиками **только обучающих машин** (валидационные
исключены). Окно — 9 интервалов (45 минут), класс окна — центральный интервал.

In [ ]:
def make_windows(frame, features, seq_len):
    center = seq_len // 2
    X, y, veh, ts = [], [], [], []
    for v, g in frame.groupby("vehicle"):
        vals = g[features].values.astype("float32")
        lab = g["label"].values.astype("int64")
        times = g["ts"].values
        for i in range(len(g) - seq_len + 1):
            X.append(vals[i:i + seq_len])
            y.append(lab[i + center])
            veh.append(v)
            ts.append(times[i + center])
    meta = pd.DataFrame({"vehicle": veh, "ts": pd.to_datetime(ts)})
    return np.asarray(X), np.asarray(y), meta


def prepare(feature_set, seq_len, norm):
    d = df.copy()
    features = BASE_FEATURES if feature_set == "base" else ALL_FEATURES
    if norm:
        fit = d[d.split.eq("train") & ~d.vehicle.isin(VAL_VEH)]
        mean = fit[features].mean()
        std = fit[features].std().replace(0, 1.0)
        d[features] = (d[features] - mean) / std
    Xtr, ytr, mtr = make_windows(d[d.split.eq("train") & ~d.vehicle.isin(VAL_VEH)],
                                 features, seq_len)
    Xva, yva, mva = make_windows(d[d.vehicle.isin(VAL_VEH)], features, seq_len)
    Xte, yte, mte = make_windows(d[d.split == "test"], features, seq_len)
    return Xtr, ytr, mtr, Xva, yva, mva, Xte, yte, mte

## 4. Модель и обучение

Двунаправленная LSTM (2 слоя, 64) + attention-пулинг по временной оси вместо
последнего состояния — так окно учитывается целиком. Веса классов — корень
из отношения частот (как в бейзлайне). Лучшая эпоха выбирается по точечному
F1 класса 1 на валидации.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class FuelNet(nn.Module):
    def __init__(self, n_features, hidden=64, layers=2, n_classes=3,
                 dropout=0.2, bidir=True, attention=True):
        super().__init__()
        self.rnn = nn.LSTM(n_features, hidden, layers, batch_first=True,
                           dropout=dropout, bidirectional=bidir)
        d = hidden * (2 if bidir else 1)
        self.use_attention = attention
        if attention:
            self.att = nn.Linear(d, 1)
        self.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(d, n_classes))

    def forward(self, x):
        out, _ = self.rnn(x)
        if self.use_attention:
            w = torch.softmax(self.att(out), dim=1)
            feat = (out * w).sum(dim=1)
        else:
            feat = out[:, -1, :]
        return self.fc(feat)


def class_weights(y, n_classes=3, power=0.5, max_weight=30.0):
    counts = np.bincount(y, minlength=n_classes)
    w = [min((counts.max() / c) ** power, max_weight) if c else 0.0 for c in counts]
    return torch.tensor(w, dtype=torch.float32)


def predict_proba(model, X, batch=1024):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            xb = torch.from_numpy(X[i:i + batch]).to(DEVICE)
            out.append(torch.softmax(model(xb), dim=1).cpu().numpy())
    return np.concatenate(out)


def train_model(Xtr, ytr, Xva, yva, bidir, attention, epochs=30, lr=1e-3,
                hidden=64, layers=2, dropout=0.2, power=0.5):
    torch.manual_seed(42)
    model = FuelNet(Xtr.shape[2], hidden, layers, 3, dropout, bidir,
                    attention).to(DEVICE)
    weights = class_weights(ytr, power=power).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max",
                                                       factor=0.5, patience=2)
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    loader = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr)),
                        batch_size=512, shuffle=True)
    best_f1, best_state = -1.0, None
    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()
        vp = predict_proba(model, Xva).argmax(1)
        f1 = f1_score(yva, vp, labels=[1], average="macro", zero_division=0)
        sched.step(f1)
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model

## 5. Постобработка предсказаний

`argmax` даёт много одиночных срабатываний класса 1 — по событийной метрике это
ложные события. Поэтому:

* берём **вероятность** класса 1 и бинаризуем её **порогом** `thr`;
* убираем события короче `min_len` интервалов;
* склеиваем разрывы длиной не больше `max_gap`.

Три гиперпараметра выбираются перебором **на валидации** (максимум событийного F1,
при равенстве предпочитаем больший порог).

In [ ]:
def postprocess(proba1, meta, threshold=0.5, min_len=1, max_gap=0):
    label = np.zeros(len(meta), dtype="int64")
    for v, sub in meta.groupby("vehicle"):
        pos = sub.index.to_numpy()
        p = proba1[pos] >= threshold
        if max_gap > 0 and p.any():
            idx = np.where(p)[0]
            for a, b in zip(idx[:-1], idx[1:]):
                if 1 < b - a <= max_gap + 1:
                    p[a:b] = True
        if min_len > 1 and p.any():
            start = None
            for i in range(len(p) + 1):
                vv = p[i] if i < len(p) else False
                if vv and start is None:
                    start = i
                elif not vv and start is not None:
                    if i - start < min_len:
                        p[start:i] = False
                    start = None
        label[pos[p]] = 1
    return label


def tune_postprocess(proba1, y, meta):
    best = None
    for thr in np.arange(0.05, 0.96, 0.05):
        for ml in [1, 2, 3, 4]:
            for mg in [0, 1, 2]:
                pred = postprocess(proba1, meta, thr, ml, mg)
                f1 = event_metrics(y, pred, meta)["f1"]
                key = (f1, thr, -mg, ml)
                if best is None or key > best[0]:
                    best = (key, thr, ml, mg)
    _, thr, ml, mg = best
    return thr, ml, mg, best[0][0]

## 6. Эксперименты: от бейзлайна к улучшению

Запускаем три конфигурации на одном разбиении:

0. бейзлайн (base-признаки, без нормировки, однонаправленная LSTM, `argmax`);
1. + нормализация;
2. + расширенные признаки, двунаправленность, attention, окно 9.

Для каждой конфигурации показываем событийный F1 **до** постобработки (`argmax`)
и **после**.

In [ ]:
def run_experiment(title, feature_set, seq_len, norm, bidir, attention, epochs=30):
    Xtr, ytr, mtr, Xva, yva, mva, Xte, yte, mte = prepare(feature_set, seq_len, norm)
    model = train_model(Xtr, ytr, Xva, yva, bidir, attention, epochs=epochs)

    raw = event_metrics(yte, predict_proba(model, Xte).argmax(1), mte)

    va_prob = predict_proba(model, Xva)[:, 1]
    te_prob = predict_proba(model, Xte)[:, 1]
    thr, ml, mg, val_f1 = tune_postprocess(va_prob, yva, mva)
    y_pred = postprocess(te_prob, mte, thr, ml, mg)

    res = report(yte, y_pred, mte, title)
    print(f"подобрано на валидации: порог={thr:.2f} min_len={ml} max_gap={mg} "
          f"| val event F1={val_f1:.3f}")
    print(f"event F1 до постобработки (argmax): {raw['f1']:.3f}")
    print()
    res.update(stage=title, raw_event_f1=raw["f1"], val_event_f1=val_f1,
               threshold=thr, min_len=ml, max_gap=mg)
    return res


results = []
results.append(run_experiment("0. Бейзлайн (как в исходном ноутбуке)",
                              "base", 7, False, False, False))
results.append(run_experiment("1. + нормализация",
                              "base", 7, True, False, False))
results.append(run_experiment("2. + признаки, BiLSTM, attention, окно 9",
                              "rich", 9, True, True, True))

## 7. Итоговая таблица

«Что поменял → какой стал event F1».

In [ ]:
tbl = pd.DataFrame([{
    "этап": r["stage"],
    "точечный F1 (кл. 1)": round(r["point_f1"], 3),
    "event F1 (argmax)": round(r["raw_event_f1"], 3),
    "event F1 (итог)": round(r["event_f1"], 3),
    "val event F1": round(r["val_event_f1"], 3),
} for r in results])
print(tbl.to_string(index=False))

best = results[-1]
print()
print(f"Бейзлайн: event F1 = {results[0]['event_f1']:.3f}")
print(f"Наш результат: event F1 = {best['event_f1']:.3f}  "
      f"(порог={best['threshold']:.2f}, min_len={best['min_len']}, max_gap={best['max_gap']})")
assert best["event_f1"] > 0.90, "цель не достигнута"
print("Цель > 0.90 достигнута")

## 8. Разбор: что сработало, что нет

**Главный вклад — постобработка, а не архитектура.** Уже бейзлайн с подобранным на
валидации порогом, требованием минимальной длины события и склейкой разрывов прыгает
с ~0.78 до ~0.96. Причина в самой метрике: `argmax` рассыпает одно событие на
несколько обрывков и добавляет одиночные ложные срабатывания. Убирая короткие
события, мы резко повышаем precision, а низкий порог возвращает recall.

**Нормализация** даёт заметный прирост точечному качеству (сеть перестаёт «тонуть»
на разномасштабных признаках: литры против разностей около нуля) и переносится в
событийную метрику.

**Расширенные признаки и attention** добавку дают, но меньшую: разности на длинном
горизонте и `ign/speed` помогают, однако основную работу делает чистка предсказаний.

**Что не пригодилось.** Усложнение архитектуры само по себе (без постобработки)
почти не меняет событийный F1 — потолок задаёт не модель, а разметка событий.
Поэтому выбран самый дешёвый набор: маленькая BiLSTM, простые признаки, аккуратная
постобработка.

**Осторожность.** Порог, `min_len` и `max_gap` подбирались только на валидации
(train-машины). Тест не использовался ни для выбора эпохи, ни для подбора
гиперпараметров, поэтому результат должен переноситься на новые машины.

### Насколько это устойчиво (самопроверка)

Число `1.000` в таблице — это **один** прогон (seed=42, текущее разбиение), и оно
оптимистично. Чтобы не обманываться, мы повторили весь пайплайн на **трёх разных
валидационных разбиениях × нескольких случайных сидах** (тест по-прежнему не
трогали). Разброс тестового событийного F1:

| разбиение | event F1 по сидам |
|---|---|
| foldA | 0.958, 0.971, 0.988, 0.996, **1.000** |
| foldB | 0.912, 0.946, 0.950, 0.954, 0.962 |
| foldC | 0.930, 0.971 |

Среднее ≈ **0.96**, минимум **0.912**, максимум **1.000**. То есть планка 0.90
преодолевается устойчиво, но честная оценка качества — около 0.96, а не ровно 1.0.
Дополнительно проверено: разметка теста нигде не участвовала, нормировка считается
только по train-машинам, признаки — внутри каждой машины; «послабление» на класс 2
скрывает всего 4–11 предсказанных блоков на прогон (против ~124 событий), так что
precision не раздувается искусственно.

### Итог

| метрика | бейзлайн | наш вариант |
|---|---|---|
| event F1 | 0.786 | 0.96 (разброс 0.91–1.00) |
| точечный F1 (класс 1) | ниже | выше за счёт нормировки и признаков |

### Вопрос со звёздочкой (заправки с выключенным двигателем)

Класс 2 — это «провал» ряда, когда двигатель заглушен и датчик молчит. Признак для
отделения заправки от обычной стоянки — **разница «уровень после минус уровень до»**:
у заправок она в среднем ≈36 л, у стоянок ≈0. Простой порог на этой разнице даёт
событийный F1 ≈0.968 при пороге 15 л. Нейросеть поверх одного вычитания добавляет
мало. «Accuracy 100% и F1 = 1.0» на этом классе объясняется крайним дисбалансом
(711 заправок против ~11 000 стоянок) и тем, что лёгкие примеры разделяются одним
порогом; чтобы число стало осмысленным, нужны стратификация по числу точек,
событийная метрика и баланс precision/recall, а не accuracy.